# Download Month-to-Date Cost Data to Lakehouse

This notebook uses the **Generate Cost Details Report** API (async) to download
cost data as a CSV file. This avoids the Query API's rate limits and 2-dimension
grouping restriction — you get **all fields** in a single request.

**How it works:**
1. POST to generate a cost details report (one API call)
2. Poll until the report is ready
3. Download the CSV from the blob URL
4. Parse into Spark DataFrames and write Delta tables

**Prerequisites:**
- A Microsoft Fabric Lakehouse attached to this notebook
- A Service Principal with **Cost Management Reader** on your subscription
- A `.env` file uploaded to Lakehouse Files (run `GetKeys.ps1` to create it)


In [ ]:
# ─── Load credentials from .env file ──────────────────────────────────────────
import os

def load_env(filepath):
    """Parse a .env file and set os.environ variables."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f".env file not found at {filepath}. "
            "Run GetKeys.ps1 first to create the service principal and .env file."
        )
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
if not os.path.exists(env_path):
    env_path = os.path.join(os.getcwd(), ".env")
if not os.path.exists(env_path):
    env_path = "/lakehouse/default/Files/.env"

load_env(env_path)

AZURE_TENANT_ID       = os.environ["AZURE_TENANT_ID"]
AZURE_CLIENT_ID       = os.environ["AZURE_CLIENT_ID"]
AZURE_CLIENT_SECRET   = os.environ["AZURE_CLIENT_SECRET"]
AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]

print(f"Loaded credentials for subscription: {AZURE_SUBSCRIPTION_ID}")
print(f"Tenant: {AZURE_TENANT_ID}")
print(f"Client: {AZURE_CLIENT_ID[:8]}...")

# Table names in Lakehouse
TABLE_COST_RECORD    = "cost_record"
TABLE_SUBSCRIPTION   = "subscription"
TABLE_RESOURCE_GROUP = "resource_group"
TABLE_RESOURCE       = "resource"
TABLE_METER          = "meter_category"
TABLE_SERVICE        = "service"


In [ ]:
import requests
import json
import time
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 1. Authenticate with Azure (OAuth2 Client Credentials)


In [ ]:
token_url = f"https://login.microsoftonline.com/{AZURE_TENANT_ID}/oauth2/v2.0/token"

token_body = {
    "grant_type":    "client_credentials",
    "client_id":     AZURE_CLIENT_ID,
    "client_secret": AZURE_CLIENT_SECRET,
    "scope":         "https://management.azure.com/.default",
}

token_response = requests.post(token_url, data=token_body)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]

print("Authentication successful.")


## 2. Generate Cost Details Report (Async API)

This uses the **Generate Cost Details Report** API instead of the Query API:
- **One API call** — no rate limit issues
- **All fields included** — no 2-dimension grouping limit
- Returns a downloadable CSV with every cost line item


In [ ]:
# ─── Submit the report request ────────────────────────────────────────────────
scope = f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
report_url = (
    f"https://management.azure.com{scope}"
    f"/providers/Microsoft.CostManagement/generateCostDetailsReport"
    f"?api-version=2024-08-01"
)

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type":  "application/json",
}

today = datetime.utcnow()
first_of_month = today.replace(day=1).strftime("%Y-%m-%d")
today_str = today.strftime("%Y-%m-%d")

report_body = {
    "metric": "ActualCost",
    "timePeriod": {
        "start": first_of_month,
        "end": today_str,
    }
}

print(f"Requesting cost report: {first_of_month} to {today_str}")
print(f"POST {report_url}")

response = requests.post(report_url, headers=headers, json=report_body)

if response.status_code == 202:
    poll_url = response.headers.get("Location")
    retry_after = int(response.headers.get("Retry-After", 30))
    print(f"\nReport generation accepted (202).")
    print(f"Retry-After: {retry_after}s")
elif response.status_code == 200:
    print("Report generated immediately (200).")
    poll_url = None
else:
    print(f"ERROR {response.status_code}")
    print(response.text)
    response.raise_for_status()


## 3. Poll for Report Completion


In [ ]:
# ─── Poll until report is ready ───────────────────────────────────────────────
download_url = None
max_polls = 60

if response.status_code == 202 and poll_url:
    for attempt in range(max_polls):
        time.sleep(retry_after)
        poll_resp = requests.get(poll_url, headers=headers)

        if poll_resp.status_code == 200:
            result = poll_resp.json()
            status = result.get("status", "")
            print(f"  Poll {attempt+1}: {status}")

            if status.lower() in ("completed", "succeeded"):
                manifest = result.get("manifest", {})
                blobs = manifest.get("blobs", [])
                if blobs:
                    download_url = blobs[0].get("blobLink")
                    byte_count = manifest.get("byteCount", 0)
                    print(f"\n  Report ready!")
                    print(f"  Blobs: {len(blobs)}")
                    print(f"  Size: {byte_count:,} bytes ({byte_count/1024/1024:.1f} MB)")
                break
            elif status.lower() == "failed":
                print(f"\n  Report FAILED:")
                print(json.dumps(result, indent=2))
                break
        elif poll_resp.status_code == 202:
            retry_after = int(poll_resp.headers.get("Retry-After", 30))
            print(f"  Poll {attempt+1}: Processing... (next check in {retry_after}s)")
        else:
            print(f"  Poll {attempt+1}: HTTP {poll_resp.status_code}")
    else:
        print("Timed out waiting for report.")
elif response.status_code == 200:
    result = response.json()
    manifest = result.get("manifest", {})
    blobs = manifest.get("blobs", [])
    if blobs:
        download_url = blobs[0].get("blobLink")

if not download_url:
    raise RuntimeError("Failed to get download URL. Check output above for errors.")

print(f"\nDownload URL obtained.")


## 4. Download the CSV


In [ ]:
# ─── Download the CSV ─────────────────────────────────────────────────────────
print("Downloading cost details CSV...")

csv_response = requests.get(download_url)
csv_response.raise_for_status()
csv_data = csv_response.content

print(f"Downloaded: {len(csv_data):,} bytes ({len(csv_data)/1024/1024:.1f} MB)")

# Save to Lakehouse Files for reference
csv_path = "/lakehouse/default/Files/cost_details_mtd.csv"
with open(csv_path, "wb") as f:
    f.write(csv_data)
print(f"Saved to: {csv_path}")


## 5. Load into Spark DataFrame


In [ ]:
# Read the CSV into Spark
cost_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/cost_details_mtd.csv")
)

row_count = cost_df.count()
col_count = len(cost_df.columns)
print(f"Loaded: {row_count:,} rows x {col_count} columns")
print(f"\nColumns:\n  {cost_df.columns}")


In [ ]:
# Preview
display(cost_df.limit(20))


## 6. Write Dimension and Fact Tables to Lakehouse

The Cost Details CSV contains ALL fields — we extract dimensions and facts from
the single DataFrame. Column names vary between EA and MCA accounts, so we
detect what's available.


In [ ]:
# ─── Detect available columns (case-insensitive) ─────────────────────────────
# The Cost Details Report CSV uses camelCase (e.g. "subscriptionName", "meterCategory")
# which differs from the Query API's PascalCase. We build a case-insensitive lookup.
available = cost_df.columns
col_map = {c.lower(): c for c in available}  # lowercase -> actual name

def pick(candidates):
    """Return the first column name found (case-insensitive), or None."""
    for c in candidates:
        actual = col_map.get(c.lower())
        if actual:
            return actual
    return None

COL_SUB_ID       = pick(["SubscriptionId", "subscriptionId"])
COL_SUB_NAME     = pick(["SubscriptionName", "subscriptionName"])
COL_RG           = pick(["ResourceGroup", "resourceGroup", "ResourceGroupName", "resourceGroupName"])
COL_RES_ID       = pick(["ResourceId", "resourceId", "InstanceId", "instanceId"])
COL_RES_TYPE     = pick(["ResourceType", "resourceType"])
COL_RES_LOC      = pick(["ResourceLocation", "resourceLocation"])
COL_METER_CAT    = pick(["MeterCategory", "meterCategory"])
COL_METER_SUB    = pick(["MeterSubCategory", "meterSubCategory"])
COL_METER_NAME   = pick(["MeterName", "meterName"])
COL_UOM          = pick(["UnitOfMeasure", "unitOfMeasure"])
COL_SVC          = pick(["ConsumedService", "consumedService"])
COL_CHARGE       = pick(["ChargeType", "chargeType"])
COL_COST         = pick(["CostInBillingCurrency", "costInBillingCurrency", "PreTaxCost", "preTaxCost", "Cost", "cost"])
COL_QTY          = pick(["Quantity", "quantity", "UsageQuantity", "usageQuantity"])
COL_DATE         = pick(["date", "Date", "UsageDateTime", "usageDateTime",
                         "servicePeriodStartDate", "ServicePeriodStartDate",
                         "servicePeriodEndDate", "ServicePeriodEndDate",
                         "BillingPeriodStartDate", "billingPeriodStartDate",
                         "UsageDate", "usageDate"])
COL_CURRENCY     = pick(["BillingCurrency", "billingCurrency", "Currency", "currency"])

# If we found a date column, check its actual data to determine parsing
if COL_DATE:
    sample_date = cost_df.select(F.col(COL_DATE)).filter(F.col(COL_DATE).isNotNull()).limit(1).collect()
    if sample_date:
        print(f"\nDate column '{COL_DATE}' sample value: '{sample_date[0][0]}' (type: {type(sample_date[0][0]).__name__})")

print("\nColumn mapping:")
for label, col in [("SubscriptionId", COL_SUB_ID), ("SubscriptionName", COL_SUB_NAME),
                   ("ResourceGroup", COL_RG), ("ResourceId", COL_RES_ID),
                   ("ResourceType", COL_RES_TYPE), ("ResourceLocation", COL_RES_LOC),
                   ("MeterCategory", COL_METER_CAT), ("MeterSubCategory", COL_METER_SUB),
                   ("MeterName", COL_METER_NAME), ("UnitOfMeasure", COL_UOM),
                   ("ConsumedService", COL_SVC), ("ChargeType", COL_CHARGE),
                   ("Cost", COL_COST), ("Quantity", COL_QTY),
                   ("Date", COL_DATE), ("Currency", COL_CURRENCY)]:
    status = "found" if col else "MISSING"
    print(f"  {label}: {col or 'N/A'} ({status})")


In [ ]:
# ─── Subscription dimension ───────────────────────────────────────────────────
if COL_SUB_ID and COL_SUB_NAME:
    sub_df = cost_df.select(
        F.col(COL_SUB_ID).alias("subscriptionId"),
        F.col(COL_SUB_NAME).alias("subscriptionName"),
    ).distinct()
else:
    sub_df = spark.createDataFrame(
        [(AZURE_SUBSCRIPTION_ID, "")], ["subscriptionId", "subscriptionName"])
sub_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_SUBSCRIPTION)
print(f"  {TABLE_SUBSCRIPTION}: {sub_df.count()} rows")

# ─── Resource Group dimension ─────────────────────────────────────────────────
if COL_RG and COL_SUB_ID:
    rg_df = cost_df.select(
        F.concat(F.col(COL_SUB_ID), F.lit("/"), F.col(COL_RG)).alias("resourceGroupId"),
        F.col(COL_RG).alias("resourceGroupName"),
        F.col(COL_SUB_ID).alias("subscriptionId"),
    ).distinct().filter(F.col(COL_RG) != "").filter(F.col(COL_RG).isNotNull())
    rg_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_RESOURCE_GROUP)
    print(f"  {TABLE_RESOURCE_GROUP}: {rg_df.count()} rows")

# ─── Resource dimension ──────────────────────────────────────────────────────
if COL_RES_ID:
    res_cols = [F.col(COL_RES_ID).alias("resourceId")]
    if COL_RES_TYPE: res_cols.append(F.col(COL_RES_TYPE).alias("resourceType"))
    if COL_RG: res_cols.append(F.col(COL_RG).alias("resourceGroupName"))
    if COL_RG and COL_SUB_ID:
        res_cols.append(F.concat(F.col(COL_SUB_ID), F.lit("/"), F.col(COL_RG)).alias("resourceGroupId"))
    if COL_RES_LOC: res_cols.append(F.col(COL_RES_LOC).alias("location"))
    res_df = cost_df.select(*res_cols).distinct().filter(F.col("resourceId") != "").filter(F.col("resourceId").isNotNull())
    res_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_RESOURCE)
    print(f"  {TABLE_RESOURCE}: {res_df.count()} rows")

# ─── Meter Category dimension ────────────────────────────────────────────────
if COL_METER_CAT:
    meter_cols = [F.col(COL_METER_CAT).alias("meterCategory")]
    if COL_METER_SUB: meter_cols.append(F.col(COL_METER_SUB).alias("meterSubCategory"))
    else: meter_cols.append(F.lit("").alias("meterSubCategory"))
    if COL_METER_NAME: meter_cols.append(F.col(COL_METER_NAME).alias("meterName"))
    else: meter_cols.append(F.lit("").alias("meterName"))
    if COL_UOM: meter_cols.append(F.col(COL_UOM).alias("unitOfMeasure"))
    else: meter_cols.append(F.lit("").alias("unitOfMeasure"))

    meter_df = cost_df.select(*meter_cols).distinct().filter(F.col("meterCategory") != "").filter(F.col("meterCategory").isNotNull())
    meter_df = meter_df.withColumn("meterId", F.md5(F.concat_ws("|",
        F.col("meterCategory"), F.col("meterSubCategory"), F.col("meterName"))))
    meter_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_METER)
    print(f"  {TABLE_METER}: {meter_df.count()} rows")

# ─── Service dimension ───────────────────────────────────────────────────────
if COL_SVC:
    svc_df = cost_df.select(
        F.col(COL_SVC).alias("serviceName"),
        F.col(COL_CHARGE).alias("chargeType") if COL_CHARGE else F.lit("Usage").alias("chargeType"),
    ).distinct().filter(F.col("serviceName") != "").filter(F.col("serviceName").isNotNull())
    svc_df = svc_df.withColumn("serviceId", F.md5(F.concat_ws("|",
        F.col("serviceName"), F.col("chargeType"))))
    svc_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_SERVICE)
    print(f"  {TABLE_SERVICE}: {svc_df.count()} rows")


In [ ]:
# ─── Cost Record fact table ──────────────────────────────────────────────────
# Only join dimension tables that were successfully created
fact_base = cost_df
has_meter = False
has_svc = False

# Check if meter_category table exists before joining
try:
    meter_lookup = spark.table(TABLE_METER).select(
        F.col("meterId").alias("_meterId"),
        F.col("meterCategory").alias("_meterCategory"),
        F.col("meterSubCategory").alias("_meterSubCategory"),
        F.col("meterName").alias("_meterName"),
    )
    if COL_METER_CAT and COL_METER_SUB and COL_METER_NAME:
        fact_base = fact_base.join(
            meter_lookup,
            (fact_base[COL_METER_CAT] == meter_lookup["_meterCategory"]) &
            (fact_base[COL_METER_SUB] == meter_lookup["_meterSubCategory"]) &
            (fact_base[COL_METER_NAME] == meter_lookup["_meterName"]),
            "left"
        )
        has_meter = True
except Exception:
    print(f"  Skipping meter join — {TABLE_METER} table not available")

# Check if service table exists before joining
try:
    svc_lookup = spark.table(TABLE_SERVICE).select(
        F.col("serviceId").alias("_serviceId"),
        F.col("serviceName").alias("_serviceName"),
        F.col("chargeType").alias("_chargeType"),
    )
    if COL_SVC and COL_CHARGE:
        fact_base = fact_base.join(
            svc_lookup,
            (fact_base[COL_SVC] == svc_lookup["_serviceName"]) &
            (fact_base[COL_CHARGE] == svc_lookup["_chargeType"]),
            "left"
        )
        has_svc = True
except Exception:
    print(f"  Skipping service join — {TABLE_SERVICE} table not available")

# Select final columns
fact_cols = [F.monotonically_increasing_id().alias("costRecordId")]

if COL_SUB_ID: fact_cols.append(F.col(COL_SUB_ID).alias("subscriptionId"))
if COL_RG and COL_SUB_ID:
    fact_cols.append(F.concat(F.col(COL_SUB_ID), F.lit("/"), F.col(COL_RG)).alias("resourceGroupId"))
if COL_RES_ID: fact_cols.append(F.col(COL_RES_ID).alias("resourceId"))
if has_meter: fact_cols.append(F.col("_meterId").alias("meterId"))
else: fact_cols.append(F.lit(None).alias("meterId"))
if has_svc: fact_cols.append(F.col("_serviceId").alias("serviceId"))
else: fact_cols.append(F.lit(None).alias("serviceId"))
if COL_DATE:
    # Try multiple date parsing approaches — the CSV format varies by account type
    date_expr = F.coalesce(
        F.to_date(F.col(COL_DATE), "MM/dd/yyyy"),         # US format: 04/15/2026
        F.to_date(F.col(COL_DATE), "yyyy-MM-dd"),          # ISO format: 2026-04-15
        F.to_date(F.col(COL_DATE), "yyyy-MM-dd'T'HH:mm:ss'Z'"),  # ISO datetime: 2026-04-15T00:00:00Z
        F.to_date(F.col(COL_DATE), "dd/MM/yyyy"),          # EU format: 15/04/2026
        F.to_date(F.col(COL_DATE)),                         # Auto-detect fallback
        F.to_date(F.substring(F.col(COL_DATE), 1, 10)),    # Take first 10 chars if datetime string
    )
    fact_cols.append(date_expr.alias("usageDate"))
if COL_COST: fact_cols.append(F.col(COL_COST).cast("double").alias("preTaxCost"))
if COL_QTY: fact_cols.append(F.col(COL_QTY).cast("double").alias("quantity"))
if COL_UOM: fact_cols.append(F.col(COL_UOM).alias("unitOfMeasure"))
if COL_CHARGE: fact_cols.append(F.col(COL_CHARGE).alias("chargeType"))
if COL_RES_LOC: fact_cols.append(F.col(COL_RES_LOC).alias("location"))
if COL_CURRENCY: fact_cols.append(F.col(COL_CURRENCY).alias("Currency"))

fact_df = fact_base.select(*fact_cols)
fact_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_COST_RECORD)
print(f"  {TABLE_COST_RECORD}: {fact_df.count():,} rows")

# Print summary of all tables that were created
print("\nTables created:")
for t in [TABLE_SUBSCRIPTION, TABLE_RESOURCE_GROUP, TABLE_RESOURCE, TABLE_METER, TABLE_SERVICE, TABLE_COST_RECORD]:
    try:
        count = spark.table(t).count()
        print(f"  ✓ {t}: {count:,} rows")
    except Exception:
        print(f"  ✗ {t}: not created (source columns missing from CSV)")


## 7. Verify — Sample Data


In [ ]:
# Summary — per resource per day cost breakdown
cost_table = spark.table(TABLE_COST_RECORD)
print(f"Cost record table: {cost_table.count():,} rows, {len(cost_table.columns)} columns")
print(f"Columns: {cost_table.columns}\n")

# Show cost per resource per day
if "resourceId" in cost_table.columns and "usageDate" in cost_table.columns:
    summary = (
        cost_table
        .filter(F.col("resourceId").isNotNull() & (F.col("resourceId") != ""))
        .groupBy("resourceId", "usageDate")
        .agg(
            F.sum("preTaxCost").alias("dailyCost"),
            F.sum("quantity").alias("totalQuantity"),
        )
        .orderBy(F.desc("dailyCost"))
    )
    print(f"Unique resource-day combinations: {summary.count():,}")
    display(summary.limit(30))
else:
    display(cost_table.limit(20))


In [ ]:
print(f"\nNotebook completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tables are ready in your Lakehouse for Power BI and Ontology creation.")
